In [1]:
# -*- coding: utf-8 -*-
"""
CTR Baseline (PyTorch) — DIN-lite + Categorical Embeddings (Notebook, OOM-safe)
- 시퀀스: 해시 임베딩(고정 버킷), Lazy 파싱(배치 시점 파싱) → 초대형 중간객체 제거
- 범주형: train 기준 factorize(OOV 처리), 연속형: BatchNorm1d
- 불균형: pos_weight = Nneg/Npos
- 지표: AUC, PR-AUC + EarlyStopping
- DataLoader: num_workers=0, pin_memory=False (노트북/Colab 안전)
"""

import os, re, json, math, random, gc, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# ---------------------
# Utils
# ---------------------
def set_seed(seed=42):
    random.seed(seed); os.environ['PYTHONHASHSEED']=str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def infer_device(arg="auto"):
    if arg=="cpu": return torch.device("cpu")
    if arg=="cuda" or (arg=="auto" and torch.cuda.is_available()):
        print("Using CUDA"); return torch.device("cuda")
    print("Using CPU"); return torch.device("cpu")

def parse_seq_string(s: str):
    """'1|2|3', '1,2,3', '[1,2,3]' 등 다양한 형태 파싱."""
    if s is None: return []
    s = str(s).strip()
    if not s or s.lower()=="nan": return []
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]
        toks = [t.strip().strip("'\"") for t in s.split(",")]
    else:
        s = re.sub(r"[^\d]+", ",", s)
        toks = [t for t in s.split(",") if t]
    out=[]
    for t in toks:
        try: out.append(int(t))
        except Exception:
            try: out.append(int(float(t)))
            except Exception: continue
    return out

def emb_dim_from_card(card, cap=64):
    return int(min(cap, max(8, round(1.6*(card**0.25)))))

# ---------------------
# Config (수정해서 쓰기)
# ---------------------
class Args:
    train = "./train.parquet"
    test  = "./test.parquet"
    label = "clicked"
    seq_col = "seq"
    id_col  = "ID"

    # 메모리/학습 설정 (대용량 안전 기본값)
    sample_subset = 1.0   # 실데이터면 먼저 2%로 파이프라인 검증 → OK면 1.0로 올리기
    max_seq_len   = 50
    HASH_BUCKETS  = 262_144  # 시퀀스 해시 임베딩 버킷(2^18). 여유있으면 524_288
    seq_emb_dim   = 64
    dropout       = 0.2

    epochs     = 3
    batch_size = 512        # GPU 여유되면 1024~2048 시도
    lr         = 1e-3
    patience   = 2

    seed = 42
    device = "auto"
    num_workers = 0         # 노트북/Colab 안전값
    pin_memory = False

args = Args()
set_seed(args.seed)
device = infer_device(args.device)

# ---------------------
# (옵션) 더미 데이터 자동 생성 - 실제 파일 있으면 건드리지 않음
# ---------------------
def create_dummy_data(num_rows, is_test=False):
    rng = np.random.default_rng(0)
    data = {
        "cont_feat1": rng.normal(size=num_rows),
        "cont_feat2": rng.random(num_rows)*10,
        "gender": rng.choice(["M","F","U"], num_rows, p=[0.45,0.45,0.1]),
        "age_group": rng.choice(["10s","20s","30s","40s+"], num_rows),
        "inventory_id": rng.integers(500, 520, num_rows),
        "day_of_week": rng.integers(0, 7, num_rows),
        "hour": rng.integers(0, 24, num_rows),
        "l_feat_14": rng.integers(1000, 1050, num_rows),
    }
    seqs=[]
    for _ in range(num_rows):
        fmt = rng.choice(["pipe","comma","json","empty","single"])
        L = int(rng.integers(1, args.max_seq_len+10))
        items = [str(int(rng.integers(1000,1500))) for _ in range(L)]
        if fmt=="pipe":   seqs.append("|".join(items))
        elif fmt=="comma":seqs.append(",".join(items))
        elif fmt=="json": seqs.append(f"[{','.join(items)}]")
        elif fmt=="empty":seqs.append(np.nan)
        else:             seqs.append(items[0])
    data[args.seq_col] = seqs
    if is_test:
        data[args.id_col] = np.arange(num_rows)
    else:
        data[args.label]  = rng.integers(0,2,num_rows)
    return pd.DataFrame(data)

if not os.path.exists(args.train):
    print("train.parquet 없음 → 소형 더미 데이터 생성(20k rows)")
    create_dummy_data(20_000, is_test=False).to_parquet(args.train)
if not os.path.exists(args.test):
    print("test.parquet 없음 → 소형 더미 데이터 생성(5k rows)")
    create_dummy_data(5_000, is_test=True).to_parquet(args.test)

# ---------------------
# 로드 + 서브샘플(메모리 안전)
# ---------------------
print("[1/6] Load parquet ...")
train = pd.read_parquet(args.train)
test  = pd.read_parquet(args.test)
assert args.label in train.columns, f"Label '{args.label}' not found"
assert args.seq_col in train.columns, f"Seq '{args.seq_col}' not found"

if 0 < args.sample_subset < 1.0:
    print(f"Subsampling to {args.sample_subset*100:.1f}%")
    train = train.sample(frac=args.sample_subset, random_state=args.seed).reset_index(drop=True)

# 사용 컬럼 정의
must_drop = {args.label, args.seq_col, args.id_col}
base_cols = [c for c in train.columns if c not in must_drop]
cand_cats = [c for c in ["gender","age_group","inventory_id","day_of_week","hour","l_feat_14"] if c in base_cols]
cont_cols = [c for c in base_cols if c not in cand_cats]
print(f"[1.1] cont={len(cont_cols)}  cat={len(cand_cats)}")

# Split (stratified)
tr, va = train_test_split(train, test_size=0.15, random_state=args.seed, stratify=train[args.label])
del train; gc.collect()

# ---------------------
# 범주형 맵(train 기준)
# ---------------------
cat_maps, cat_cards = {}, {}
for c in cand_cats:
    cats = pd.Categorical(tr[c])
    cat_maps[c]  = {v:i for i,v in enumerate(cats.categories)}
    cat_cards[c] = len(cats.categories)

# ---------------------
# 시퀀스: 해시 임베딩 (vocab X)
# ---------------------
PAD_ID, OOV_ID = 0, 1
SEQ_BASE = 2
seq_vocab_size = args.HASH_BUCKETS + SEQ_BASE  # 0:PAD, 1:OOV, 2..hash

def seq_to_ids_hash(lst, max_len=50):
    # 정수 토큰 t → 2 + (t % HASH_BUCKETS)
    ids = [SEQ_BASE + (int(t) % args.HASH_BUCKETS) for t in lst][-max_len:]
    if len(ids) < max_len:
        ids = [PAD_ID]*(max_len-len(ids)) + ids
    return np.array(ids, dtype=np.int32)

# ---------------------
# Dataset (Lazy 파싱)
# ---------------------
class CTRDataset(Dataset):
    def __init__(self, df, cont_cols, cat_cols, cat_maps, seq_col, max_seq_len, label_col=None):
        self.df = df.reset_index(drop=True)
        self.cont_cols, self.cat_cols = cont_cols, cat_cols
        self.cat_maps, self.seq_col = cat_maps, seq_col
        self.max_seq_len = max_seq_len
        self.has_label = label_col is not None
        self.label_col = label_col
        # 연속형/범주형 값만 저장 (작음)
        self.Xc = self.df[self.cont_cols].astype(np.float32).fillna(0.0).values if self.cont_cols else None
        self.Xcats = {c: self.df[c].map(self.cat_maps[c]).fillna(cat_cards[c]).astype(np.int64).values
                      for c in self.cat_cols}  # OOV: len(categories)
        if self.has_label:
            self.y = self.df[self.label_col].astype(np.float32).values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        xc = torch.from_numpy(self.Xc[idx]) if self.Xc is not None else torch.empty(0)
        cats = {c: torch.tensor(self.Xcats[c][idx], dtype=torch.long) for c in self.cat_cols}

        s = self.df.at[idx, self.seq_col]
        lst = parse_seq_string(s)              # !!! 그때그때 파싱
        seq_ids = seq_to_ids_hash(lst, self.max_seq_len)
        seq = torch.from_numpy(seq_ids).long()

        if self.has_label:
            return xc, cats, seq, torch.tensor(self.y[idx], dtype=torch.float32)
        else:
            return xc, cats, seq

# ---------------------
# Model (DIN-lite)
# ---------------------
class DINModel(nn.Module):
    def __init__(self, cont_dim, cat_cards, seq_vocab_size, target_name=None,
                 seq_emb_dim=64, mlp_units=[512,256,128], dropout=0.2):
        super().__init__()
        self.has_cont = cont_dim > 0
        if self.has_cont:
            self.bn = nn.BatchNorm1d(cont_dim)

        # categorical embeddings
        self.cat_embs = nn.ModuleDict()
        for name, card in cat_cards.items():
            dim = emb_dim_from_card(card+1)
            self.cat_embs[name] = nn.Embedding(card+1+1, dim)
        cat_dim = sum(emb.embedding_dim for emb in self.cat_embs.values())

        # sequence embedding
        self.seq_emb = nn.Embedding(seq_vocab_size, seq_emb_dim, padding_idx=PAD_ID)

        # --- NEW: target proj to match seq_emb_dim ---
        self.target_name = target_name if (target_name in self.cat_embs) else None
        if self.target_name is not None:
            target_dim = self.cat_embs[self.target_name].embedding_dim
        else:
            target_dim = seq_emb_dim
        self.proj_t = nn.Linear(target_dim, seq_emb_dim, bias=False) if target_dim != seq_emb_dim else nn.Identity()
        # ------------------------------------------------

        in_dim = (cont_dim if self.has_cont else 0) + cat_dim + seq_emb_dim
        layers=[]
        for h in mlp_units:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.mlp = nn.Sequential(*layers)


    def forward(self, xc, xcats, seq_ids):
        outs=[]
        if self.has_cont:
            outs.append(self.bn(xc))
        if len(self.cat_embs)>0:
            outs.append(torch.cat([self.cat_embs[n](xcats[n]) for n in self.cat_embs], dim=1))

        seq_vec = self.seq_emb(seq_ids)  # (B,L,Dseq)

        if self.target_name is not None:
            t_vec = self.cat_embs[self.target_name](xcats[self.target_name])  # (B,Dt)
            t_vec = self.proj_t(t_vec)  # <-- D_t -> D_seq 맞추기 (핵심)
        else:
            mask = (seq_ids != PAD_ID).float().unsqueeze(-1)
            t_vec = (seq_vec * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)

        # DIN dot-attention
        att = (seq_vec * t_vec.unsqueeze(1)).sum(dim=2)        # (B,L)
        mask = (seq_ids != PAD_ID).float()
        att = att.masked_fill(mask == 0, -1e9)
        att = torch.softmax(att, dim=1)
        att = torch.nan_to_num(att, nan=0.0)  # <-- 전부 PAD인 행의 NaN 방지

        interest = torch.sum(seq_vec * att.unsqueeze(2), dim=1)  # (B,Dseq)

        outs.append(interest)
        z = torch.cat(outs, dim=1)
        return self.mlp(z).squeeze(1)

# ---------------------
# DataLoaders (안전 설정)
# ---------------------
target_name = "l_feat_14" if "l_feat_14" in cand_cats else None
ds_tr = CTRDataset(tr, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_va = CTRDataset(va, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=args.label)
ds_te = CTRDataset(test, cont_cols, cand_cats, cat_maps, args.seq_col, args.max_seq_len, label_col=None)

n_pos = float((tr[args.label]==1).sum()); n_neg = float(len(tr)-n_pos)
pos_weight = torch.tensor(max(1.0, n_neg/max(1.0,n_pos)), dtype=torch.float32, device=device)
print(f"[2/6] Train label: pos={int(n_pos)} neg={int(n_neg)}  pos_weight={pos_weight.item():.3f}")

dl_tr = DataLoader(ds_tr, batch_size=args.batch_size, shuffle=True,
                   num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_va = DataLoader(ds_va, batch_size=args.batch_size, shuffle=False,
                   num_workers=args.num_workers, pin_memory=args.pin_memory)
dl_te = DataLoader(ds_te, batch_size=args.batch_size, shuffle=False,
                   num_workers=args.num_workers, pin_memory=args.pin_memory)

# ---------------------
# Train
# ---------------------
print("[3/6] Build model ...")
model = DINModel(cont_dim=len(cont_cols), cat_cards=cat_cards, seq_vocab_size=seq_vocab_size,
                 target_name=target_name, seq_emb_dim=args.seq_emb_dim,
                 mlp_units=[512,256,128], dropout=args.dropout).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
scaler = GradScaler(enabled=(device.type=="cuda"))

best_auc, best_state, wait = -1.0, None, 0
final_epoch, final_prauc = 0, 0.0

print("[4/6] Train ...")
for epoch in range(1, args.epochs+1):
    model.train(); tr_loss=0.0
    for xc, cats, seq, y in dl_tr:
        xc, seq, y = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True), y.to(device, non_blocking=True)
        cats_dev = {k: v.to(device, non_blocking=True) for k,v in cats.items()}

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            logits = model(xc, cats_dev, seq)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*y.size(0)
    tr_loss /= len(ds_tr); scheduler.step()

    # Val
    model.eval(); va_loss=0.0; ys=[]; ps=[]
    with torch.no_grad():
        for xc, cats, seq, y in dl_va:
            xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
            cats_dev = {k: v.to(device, non_blocking=True) for k,v in cats.items()}
            with autocast(enabled=(device.type=="cuda")):
                logits = model(xc, cats_dev, seq)
                loss = criterion(logits, y.to(device))
                prob = torch.sigmoid(logits).detach().cpu().numpy()
            va_loss += loss.item()*len(y)
            ys.append(y.numpy()); ps.append(prob)
    va_loss /= len(ds_va)
    y_true = np.concatenate(ys); y_prob = np.concatenate(ps)
    auc = roc_auc_score(y_true, y_prob)
    prauc = average_precision_score(y_true, y_prob)
    print(f"Epoch {epoch:02d} | Train {tr_loss:.5f} | Val {va_loss:.5f} | AUC {auc:.6f} | PR-AUC {prauc:.6f}")

    if auc > best_auc + 1e-5:
        best_auc, final_prauc, final_epoch = auc, prauc, epoch
        best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= args.patience:
            print(f"Early stopping. Best AUC={best_auc:.6f} at epoch {final_epoch}.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

# ---------------------
# Inference
# ---------------------
print("[5/6] Predict test ...")
model.eval(); probs=[]
with torch.no_grad():
    for xc, cats, seq in dl_te:
        xc, seq = xc.to(device, non_blocking=True), seq.to(device, non_blocking=True)
        cats_dev = {k: v.to(device, non_blocking=True) for k,v in cats.items()}
        with autocast(enabled=(device.type=="cuda")):
            logits = model(xc, cats_dev, seq)
            prob = torch.sigmoid(logits).cpu().numpy()
        probs.append(prob)
probs = np.concatenate(probs)

# ---------------------
# Save
# ---------------------
sub_path = "./din_baseline_submit.csv"
meta_path= "./din_meta.json"
sub = pd.DataFrame({args.id_col: test[args.id_col].values, "clicked": probs})
sub.to_csv(sub_path, index=False)

meta = {
    "columns": {"continuous": cont_cols, "categorical": cand_cats},
    "seq_vocab": {"type": "hash", "buckets": args.HASH_BUCKETS, "pad_id": 0, "oov_id": 1},
    "hyperparameters": {
        "sample_subset": args.sample_subset, "max_seq_len": args.max_seq_len,
        "seq_emb_dim": args.seq_emb_dim, "dropout": args.dropout,
        "epochs": args.epochs, "batch_size": args.batch_size, "lr": args.lr,
    },
    "performance": {"best_epoch": int(final_epoch), "AUC": float(best_auc), "PR_AUC": float(final_prauc)}
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("\n[6/6] Done. Outputs:")
print(f" - {sub_path}")
print(f" - {meta_path}")


Using CUDA
[1/6] Load parquet ...
[1.1] cont=111  cat=6
[2/6] Train label: pos=173552 neg=8925000  pos_weight=51.426
[3/6] Build model ...
[4/6] Train ...
Epoch 01 | Train 1.20945 | Val 1.19284 | AUC 0.736693 | PR-AUC 0.073800
Epoch 02 | Train 1.18795 | Val 1.19334 | AUC 0.739313 | PR-AUC 0.075393
Epoch 03 | Train 1.17534 | Val 1.21143 | AUC 0.741223 | PR-AUC 0.077857
[5/6] Predict test ...

[6/6] Done. Outputs:
 - ./din_baseline_submit.csv
 - ./din_meta.json
